# Laboratorio: el espejismo del 99 % de accuracy

**Pregunta:** ¿por qué un 99 % de accuracy no implica que un detector de intrusiones funcione?

**Hipótesis (fijada antes de medir):** un accuracy ≥ 99 % es compatible con un detector inútil, o incluso peor que no hacer nada, y no distingue entre detectores que se comportan de forma muy distinta (falsos negativos, falsos positivos, recall y precisión).

**Diseño:**
- Mismo dataset (UNSW-NB15), mismas variables y mismo modelo que en [Azar vs. tiempo](../Azar-Tiempo/): regresión logística con umbral 0,5.
- Se entrena **una vez por escenario** (reparto por tiempo A → B y reparto al azar) y se guardan las curvas TPR/FPR para todos los umbrales.
- Lo que se hace variar es la **prevalencia de ataques** (cuántos ataques hay en la prueba) y el **umbral**. Como TPR y FPR no dependen de la prevalencia, las métricas para cualquier prevalencia se calculan con exactitud a partir de ellas; después se comprueba con re-muestreo real.
- Se compara siempre contra detectores «tontos» de referencia (todo normal, todo ataque, al azar).

**Reglas de decisión (C1, C2, C3):** están fijadas en la celda de configuración y en el informe, antes de ejecutar.

**Importante:** los umbrales de las secciones 4 y 5 se eligen mirando las etiquetas de prueba. Sirven para **ilustrar puntos de operación**, no para seleccionar un modelo.

## 0. Configuración (decide esto ANTES de ejecutar)

In [ ]:
import time, warnings, math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings("ignore")
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 30)

# ---- Parámetros del experimento ------------------------------------------
DATA_DIR = Path("UNSW-NB15")   # carpeta con UNSW-NB15_1.csv ... UNSW-NB15_4.csv
FIG_DIR = Path("figuras")      # aquí se guardan las figuras
SAMPLE_FRAC = 1.0              # 1.0 = todos los datos (lento). 0.25 para probar rápido.
N_RANDOM_REPEATS = 5           # repeticiones del reparto al azar
THRESHOLD = 0.5                # umbral "por defecto" (el mismo del subcapítulo anterior)
PREVALENCES = [0.5, 0.2, 0.1, 0.05, 0.01, 0.001]   # prevalencias simuladas de ataques en la prueba
P_MAIN = 0.01                  # prevalencia principal: aquella en la que "todo normal" ya saca 99 %
N_RESAMPLES = 10               # re-muestreos reales para verificar el cálculo exacto

# ---- Reglas de decisión (fíjalas antes de mirar resultados) ----------------
ACC_MIRAGE = 0.99          # "el 99 %": accuracy a partir del cual se declara el espejismo
ACC_TIE = 0.005            # C1: |accuracy del modelo - accuracy de "todo normal"| < 0,5 puntos = EMPATE.
                           #     A prevalencia 1 % la ganancia máxima posible es 1 punto (modelo perfecto):
                           #     un empate es no llegar a eliminar ni la mitad de los errores del que no hace nada.
MIN_RECALL_USEFUL = 0.5    # C1: el modelo se considera útil si detecta al menos la mitad de los ataques
RECALL_RANGE_BIG = 0.30    # C2: entre los umbrales con accuracy >= ACC_MIRAGE (y recall > 0), el recall debe variar >= 30 puntos
ACC_OPT_DROP = 0.05        # C3: el umbral que maximiza accuracy debe perder >= 5 puntos de recall frente al umbral 0,5
MIN_FRAC_AGREE = 0.75      # fracción de escenarios en los que debe cumplirse cada regla (con 2 escenarios: 2 de 2)
SEED = 42

# ---- REVISIÓN EXPLORATORIA (añadida DESPUÉS de ver los datos; no es parte del pre-registro) ----
C2B_RECALL_FLOOR = 0.5                       # C2': el detector debe detectar al menos la mitad de los ataques (mismo criterio de "útil" que C1)
C2B_FLOORS_SENS = [0.0, 0.1, 0.3, 0.5, 0.7, 0.9]   # suelos de recall para ver cuánto depende la conclusión de esta elección arbitraria
ACC_OPT_DROP_VS_F1 = 0.05                    # C3': el umbral que maximiza accuracy debe perder >= 5 puntos de recall frente al que maximiza F1
RES_DIR = Path("resultados")                 # aquí se exportan las tablas en CSV para estudiarlas

## 1. Carga y limpieza

Mismo dataset y mismas decisiones que en el subcapítulo anterior: **UNSW-NB15**, 4 CSV en bruto sin cabecera. Se excluyen IP, puertos y marcas de tiempo como variables; el tiempo solo sirve para ordenar y repartir.

In [ ]:
COLS = ["srcip","sport","dstip","dsport","proto","state","dur","sbytes","dbytes","sttl","dttl",
        "sloss","dloss","service","Sload","Dload","Spkts","Dpkts","swin","dwin","stcpb","dtcpb",
        "smeansz","dmeansz","trans_depth","res_bdy_len","Sjit","Djit","Stime","Ltime","Sintpkt",
        "Dintpkt","tcprtt","synack","ackdat","is_sm_ips_ports","ct_state_ttl","ct_flw_http_mthd",
        "is_ftp_login","ct_ftp_cmd","ct_srv_src","ct_srv_dst","ct_dst_ltm","ct_src_ltm",
        "ct_src_dport_ltm","ct_dst_sport_ltm","ct_dst_src_ltm","attack_cat","Label"]
assert len(COLS) == 49

ID_COLS    = ["srcip", "sport", "dstip", "dsport"]   # identificadores: fuera del modelo
TIME_COLS  = ["Stime", "Ltime"]                      # el tiempo sirve para repartir, NO como variable
LABEL_COLS = ["Label", "attack_cat"]
CAT_COLS   = ["proto", "state", "service"]
NUM_COLS   = [c for c in COLS if c not in ID_COLS + TIME_COLS + LABEL_COLS + CAT_COLS]
FEAT_COLS  = NUM_COLS + CAT_COLS


def load_unsw(data_dir):
    files = sorted(Path(data_dir).glob("UNSW-NB15_[1-4].csv"))
    assert len(files) == 4, f"Esperaba 4 ficheros UNSW-NB15_[1-4].csv en {data_dir}, encontré {len(files)}"
    return pd.concat([pd.read_csv(f, header=None, names=COLS, low_memory=False) for f in files],
                     ignore_index=True)


df = load_unsw(DATA_DIR)
print("Filas cargadas:", len(df))

for c in ["sport", "dsport", "ct_ftp_cmd", "ct_flw_http_mthd", "is_ftp_login", "Stime", "Ltime"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
for c in CAT_COLS + ["srcip", "dstip"]:
    df[c] = df[c].astype(str).str.strip()
df["attack_cat"] = df["attack_cat"].fillna("Normal").astype(str).str.strip().replace({"": "Normal", "Backdoors": "Backdoor"})
df["Label"] = pd.to_numeric(df["Label"], errors="coerce").astype(int)
df = df.dropna(subset=["Stime"])

# orden temporal (la base del reparto por tiempo)
df = df.sort_values("Stime", kind="mergesort").reset_index(drop=True)

if SAMPLE_FRAC < 1.0:   # submuestreo uniforme: conserva el orden temporal
    df = df.sample(frac=SAMPLE_FRAC, random_state=SEED).sort_values("Stime", kind="mergesort").reset_index(drop=True)

y = df["Label"].astype(int)
n = len(df)
print(f"Filas usadas: {n:,} | ataques: {y.sum():,} ({y.mean():.1%})")

### 1.1 Las dos capturas y la línea base «todo normal»

Antes de hablar del 99 %, hay que mirar **cuánto saca el detector que no detecta nada** en estos datos. Si en ninguna parte llega al 99 %, el espejismo no aparece «de forma natural» y habrá que simular prevalencias más realistas (sección 2).

In [ ]:
t = pd.to_datetime(df["Stime"], unit="s")
st = df["Stime"].values
gaps = np.diff(st)
assert gaps.max() > 24 * 3600, "No detecto dos capturas separadas por un hueco grande"
split_pos = int(np.argmax(gaps)) + 1            # primera fila de la captura B
n_A = split_pos

caps = []
for name, lo, hi in [("A", 0, split_pos), ("B", split_pos, n)]:
    yy = y.iloc[lo:hi]
    caps.append(dict(captura=name, desde=t.iloc[lo], hasta=t.iloc[hi - 1],
                     filas=hi - lo, ataques=int(yy.sum()), pct_ataques=yy.mean() * 100,
                     acc_todo_normal=(1 - yy.mean()) * 100))
caps.append(dict(captura="Total", desde=t.iloc[0], hasta=t.iloc[-1], filas=n, ataques=int(y.sum()),
                 pct_ataques=y.mean() * 100, acc_todo_normal=(1 - y.mean()) * 100))
display(pd.DataFrame(caps).round(2))
print(f"\nPara que 'todo normal' saque un {ACC_MIRAGE:.0%} de accuracy, la prevalencia de ataques tendría que ser del {1 - ACC_MIRAGE:.0%}.")

## 2. Piezas del experimento

- **Modelo:** regresión logística con el mismo preprocesado de siempre (mediana, `log1p`, estandarizado, one-hot).
- **Métricas calculadas a mano**, con la regla de siempre: si una división es 0/0, el resultado es `NaN` («sin definir»), nunca 0. Importante aquí: un detector que **no lanza ninguna alerta no tiene precisión** (0/0).
- **Curvas TPR/FPR para toda una rejilla de umbrales**, y a partir de ellas las métricas **exactas** para cualquier prevalencia:

```
TP = TPR · p      FN = (1 − TPR) · p      FP = FPR · (1 − p)      TN = (1 − FPR) · (1 − p)
```
- **Re-muestreo real** (para verificar que el cálculo exacto no falla): se construye un conjunto de prueba con la prevalencia deseada submuestreando ataques o normales.

In [ ]:
def log1p_pos(a):
    return np.log1p(np.clip(a, 0, None))


def make_model(num_cols, cat_cols, min_freq=100):
    parts = []
    if num_cols:
        parts.append(("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                                       ("log", FunctionTransformer(log1p_pos, feature_names_out="one-to-one")),
                                       ("sc", StandardScaler())]), list(num_cols)))
    if cat_cols:
        parts.append(("cat", OneHotEncoder(handle_unknown="infrequent_if_exist",
                                           min_frequency=min_freq), list(cat_cols)))
    return Pipeline([("prep", ColumnTransformer(parts)),
                     ("clf", LogisticRegression(max_iter=1000))])


def counts_metrics(tp, fp, fn, tn):
    # Sirve con conteos o con fracciones del total. Divisiones 0/0 -> NaN ("sin definir").
    tp, fp, fn, tn = float(tp), float(fp), float(fn), float(tn)
    P, N = tp + fn, fp + tn
    tot = P + N
    div = lambda a, b: a / b if b > 0 else np.nan
    recall, fpr = div(tp, P), div(fp, N)
    den = (tp + fp) * (tp + fn) * (tn + fp) * (tn + fn)
    return dict(
        accuracy=(tp + tn) / tot,
        precision=div(tp, tp + fp),           # de mis alertas, cuántas eran ataques (sin alertas: sin definir)
        recall=recall,                        # de los ataques reales, cuántos capturé
        f1=div(2 * tp, 2 * tp + fp + fn),
        fpr=fpr,                              # de las conexiones normales, cuántas disparan alarma
        fnr=div(fn, P),                       # de los ataques reales, cuántos se escapan
        balanced_acc=(recall + (1 - fpr)) / 2,
        mcc=(tp * tn - fp * fn) / math.sqrt(den) if den > 0 else np.nan,
        prevalence=P / tot,
        acc_todo_normal=N / tot,
    )


def metrics_from_proba(y_true, proba, thr=THRESHOLD):
    pred = np.asarray(proba) >= thr
    yt = np.asarray(y_true).astype(bool)
    return counts_metrics((pred & yt).sum(), (pred & ~yt).sum(), (~pred & yt).sum(), (~pred & ~yt).sum())


def roc_points(y_true, proba, thr_grid):
    # TPR y FPR para cada umbral (alerta si proba >= umbral), sin recorrer los datos una vez por umbral.
    y_true = np.asarray(y_true).astype(bool)
    pos, neg = np.sort(proba[y_true]), np.sort(proba[~y_true])
    tpr = 1 - np.searchsorted(pos, thr_grid, side="left") / len(pos)
    fpr = 1 - np.searchsorted(neg, thr_grid, side="left") / len(neg)
    return tpr, fpr


# Rejilla de umbrales. El último (> 1) es el detector que NO alerta nunca ("silencio total").
THR_GRID = np.unique(np.concatenate([np.linspace(0, 1, 1001), [0.9995, 0.9999, 1.0000001]]))
I05 = int(np.argmin(np.abs(THR_GRID - THRESHOLD)))


def at_prevalence(tpr, fpr, p):
    # Métricas exactas para una prevalencia p, a partir de TPR y FPR (que no dependen de p).
    m = counts_metrics(tpr * p, fpr * (1 - p), (1 - tpr) * p, (1 - fpr) * (1 - p))
    m.update(tp_mill=tpr * p * 1e6, fn_mill=(1 - tpr) * p * 1e6,
             fp_mill=fpr * (1 - p) * 1e6, tn_mill=(1 - fpr) * (1 - p) * 1e6)
    return m


def resample_to_prevalence(y_true, p, rng):
    # Índices de un conjunto de prueba con prevalencia p: submuestrea ataques si sobran normales, y al revés.
    y_true = np.asarray(y_true)
    pos, neg = np.flatnonzero(y_true == 1), np.flatnonzero(y_true == 0)
    need_pos = p / (1 - p) * len(neg)
    if need_pos <= len(pos):
        keep_pos, keep_neg = rng.choice(pos, size=int(round(need_pos)), replace=False), neg
    else:
        keep_pos, keep_neg = pos, rng.choice(neg, size=int(round(len(pos) * (1 - p) / p)), replace=False)
    return np.concatenate([keep_pos, keep_neg])

## 3. Entrenar y medir en dos escenarios

| Escenario | Entrenamiento | Prueba | Repeticiones |
|---|---|---|---|
| **tiempo (A → B)** | toda la captura A | toda la captura B | 1 (determinista) |
| **azar** | mismas filas que A, elegidas al azar de todo el dataset | las demás (mismo tamaño que B) | `N_RANDOM_REPEATS` |

Son los mismos escenarios del «caso duro» del subcapítulo anterior, para poder compararlos. De cada uno se guardan las **curvas TPR/FPR** y la prevalencia del entrenamiento.

In [ ]:
def fit_proba(tr, te):
    m = make_model(NUM_COLS, CAT_COLS)
    m.fit(df.iloc[tr][FEAT_COLS], y.iloc[tr])
    return m.predict_proba(df.iloc[te][FEAT_COLS])[:, 1]


def make_splits(name):
    if name == "tiempo (A → B)":
        return [(np.arange(0, n_A), np.arange(n_A, n))]
    out = []
    for r in range(N_RANDOM_REPEATS):
        perm = np.random.default_rng(SEED + r).permutation(n)
        out.append((perm[:n_A], perm[n_A:]))
    return out


scen = {}
t0 = time.time()
for name in ["tiempo (A → B)", "azar"]:
    tprs, fprs, qs = [], [], []
    for r, (tr, te) in enumerate(make_splits(name)):
        assert y.iloc[tr].nunique() == 2, "el entrenamiento necesita ataques y tráfico normal"
        proba = fit_proba(tr, te)
        yt = y.iloc[te].values
        tpr, fpr = roc_points(yt, proba, THR_GRID)
        tprs.append(tpr); fprs.append(fpr); qs.append(float(y.iloc[tr].mean()))
        if r == 0:
            first = dict(y_te=yt, proba=proba)
        print(f"{name} · repetición {r} lista ({time.time() - t0:.0f} s)")
    k = len(tprs)
    scen[name] = dict(tpr=np.mean(tprs, 0), fpr=np.mean(fprs, 0),
                      tpr_std=np.std(tprs, 0, ddof=1) if k > 1 else np.zeros(len(THR_GRID)),
                      fpr_std=np.std(fprs, 0, ddof=1) if k > 1 else np.zeros(len(THR_GRID)),
                      q_train=float(np.mean(qs)), n_runs=k, n_test=len(te), n_test_pos=int(yt.sum()), first=first)

rows = []
for name, s in scen.items():
    rows.append(dict(escenario=name, repeticiones=s["n_runs"], prevalencia_prueba=s["n_test_pos"] / s["n_test"],
                     prevalencia_entrenamiento=s["q_train"],
                     TPR_umbral05=s["tpr"][I05], TPR_std=s["tpr_std"][I05],
                     FPR_umbral05=s["fpr"][I05], FPR_std=s["fpr_std"][I05]))
display(pd.DataFrame(rows).round(4))

### 3.1 El espejismo: el mismo modelo con distinta prevalencia

Con el **mismo modelo y el mismo umbral**, solo cambia cuántos ataques hay en la prueba. La columna `ganancia` es `accuracy del modelo − accuracy de «todo normal»` (= `1 − p`). Si la ganancia es ≤ 0, el detector que no detecta nada puntúa igual o mejor que el modelo.

In [ ]:
rows = []
for name, s in scen.items():
    for p in PREVALENCES:
        m = at_prevalence(s["tpr"][I05], s["fpr"][I05], p)
        rows.append(dict(escenario=name, prevalencia=p, acc_modelo=m["accuracy"], acc_todo_normal=1 - p,
                         ganancia=m["accuracy"] - (1 - p), recall=m["recall"], precision=m["precision"],
                         fpr=m["fpr"], f1=m["f1"], mcc=m["mcc"],
                         FP_por_millon=m["fp_mill"], FN_por_millon=m["fn_mill"]))
tab_prev = pd.DataFrame(rows)
display(tab_prev.round(4))

# --- Detectores tontos de referencia a la prevalencia principal ---
print(f"\nDetectores de referencia con prevalencia {P_MAIN:.1%} (valores por millón de conexiones donde se indica)")
rows = []
for name, s in scen.items():
    q = s["q_train"]
    contenders = {"modelo (umbral 0,5)": (s["tpr"][I05], s["fpr"][I05]),
                  "todo normal": (0.0, 0.0),
                  "todo ataque": (1.0, 1.0),
                  "al azar (prior del entrenamiento)": (q, q)}
    for who, (tpr, fpr) in contenders.items():
        m = at_prevalence(tpr, fpr, P_MAIN)
        rows.append(dict(escenario=name, detector=who, accuracy=m["accuracy"], recall=m["recall"],
                         precision=m["precision"], fpr=m["fpr"], f1=m["f1"], mcc=m["mcc"],
                         TP=m["tp_mill"], FN=m["fn_mill"], FP=m["fp_mill"]))
display(pd.DataFrame(rows).round(4))

### 3.2 Regla C1: ¿el accuracy distingue al modelo del detector que no hace nada?

Con prevalencia `P_MAIN` y umbral 0,5, para cada escenario se calcula `Δ = accuracy del modelo − accuracy de «todo normal»`:

- `|Δ| < ACC_TIE` → **EMPATE** (el accuracy no ve la diferencia). A prevalencia `p` la ganancia máxima posible es `p` (modelo perfecto), así que con `p = 1 %` y `ACC_TIE = 0,5 puntos`, empatar significa **no eliminar ni la mitad de los errores** del detector que no hace nada
- `Δ ≤ −ACC_TIE` → **EL TONTO GANA** (el accuracy prefiere al detector que no detecta nada)
- `Δ ≥ +ACC_TIE` → **EL MODELO GANA CLARO**

La hipótesis se apoya si, siendo el modelo útil (recall ≥ `MIN_RECALL_USEFUL`), el resultado es EMPATE o EL TONTO GANA en la fracción exigida de escenarios. Hay además una identidad que se comprueba en el código: **`Δ = TP − FP`** (en fracciones del total).

In [ ]:
def need_n(k):
    return max(1, math.ceil(MIN_FRAC_AGREE * k))


def veredicto_c1():
    hits, k = 0, len(scen)
    for name, s in scen.items():
        tpr, fpr = s["tpr"][I05], s["fpr"][I05]
        m = at_prevalence(tpr, fpr, P_MAIN)
        delta = m["accuracy"] - (1 - P_MAIN)
        assert abs(delta - (tpr * P_MAIN - fpr * (1 - P_MAIN))) < 1e-12       # Δ = TP − FP
        if delta <= -ACC_TIE:   v = "EL TONTO GANA"
        elif delta >= ACC_TIE:  v = "EL MODELO GANA CLARO"
        else:                   v = "EMPATE"
        useful = m["recall"] >= MIN_RECALL_USEFUL
        apoya = useful and v in ("EMPATE", "EL TONTO GANA")
        hits += int(apoya)
        print(f"{name:>15}: acc modelo {m['accuracy']:.4f} | acc 'todo normal' {1 - P_MAIN:.4f} | Δ = {delta:+.4f} "
              f"| recall {m['recall']:.3f} (útil: {useful}) | precisión {m['precision']:.3f} -> {v} | apoya la hipótesis: {apoya}")
    ok = hits >= need_n(k)
    print(f"\nC1 se cumple en {hits}/{k} escenarios (se exigen {need_n(k)}) -> {'SE CUMPLE' if ok else 'NO SE CUMPLE'}")
    return ok

C1_OK = veredicto_c1()

## 4. Regla C2: mismo accuracy, detectores muy distintos

Con prevalencia `P_MAIN`, se recorren todos los umbrales del **mismo modelo**. Se seleccionan los que dan `accuracy ≥ ACC_MIRAGE` **y recall > 0** (se excluye el detector que no alerta nunca, para que la regla no sea trivial). Si dentro de ese grupo el recall varía en `RECALL_RANGE_BIG` puntos o más, el accuracy no distingue entre detectores muy diferentes.

> Los umbrales se eligen mirando la prueba: es una ilustración de puntos de operación, no una selección de modelo.

In [ ]:
def curve_table(s, p):
    rows = []
    for i, thr in enumerate(THR_GRID):
        m = at_prevalence(s["tpr"][i], s["fpr"][i], p)
        rows.append(dict(umbral=thr, accuracy=m["accuracy"], recall=m["recall"], precision=m["precision"],
                         fpr=m["fpr"], f1=m["f1"], FP_por_millon=m["fp_mill"], FN_por_millon=m["fn_mill"]))
    return pd.DataFrame(rows)

curves = {name: curve_table(s, P_MAIN) for name, s in scen.items()}


def veredicto_c2():
    hits, k = 0, len(scen)
    for name, tb in curves.items():
        S = tb[(tb.accuracy >= ACC_MIRAGE) & (tb.recall > 0)]
        if len(S) < 2:
            print(f"{name:>15}: solo {len(S)} umbral(es) con accuracy ≥ {ACC_MIRAGE:.0%} y recall > 0 -> NO se puede evaluar el rango (cuenta como no cumplida)")
            continue
        rng_ = S.recall.max() - S.recall.min()
        ok = rng_ >= RECALL_RANGE_BIG
        hits += int(ok)
        print(f"{name:>15}: {len(S)} umbrales con accuracy ≥ {ACC_MIRAGE:.0%} | recall de {S.recall.min():.3f} a {S.recall.max():.3f} "
              f"(rango {rng_:.3f}) | precisión de {S.precision.min():.3f} a {S.precision.max():.3f} -> {'SE CUMPLE' if ok else 'no se cumple'}")
    ok_all = hits >= need_n(k)
    print(f"\nC2 se cumple en {hits}/{k} escenarios (se exigen {need_n(k)}) -> {'SE CUMPLE' if ok_all else 'NO SE CUMPLE'}")
    return ok_all

C2_OK = veredicto_c2()

print("\nUmbrales de ejemplo (prevalencia %.1f%%):" % (P_MAIN * 100))
sel = [0.01, 0.05, 0.1, 0.3, 0.5, 0.7, 0.9, 0.99, 0.999, 1.0000001]
for name, tb in curves.items():
    print(f"\n--- {name} ---")
    display(tb.iloc[[int(np.argmin(np.abs(THR_GRID - u))) for u in sel]].round(4).reset_index(drop=True))

## 5. Regla C3: optimizar el accuracy empuja al silencio

Con prevalencia baja, cada falsa alarma «cuesta» lo mismo que un ataque no detectado, pero hay muchísimas más conexiones normales que ataques. Optimizar el accuracy invita a **alertar menos**. La regla: el umbral que maximiza el accuracy debe tener al menos `ACC_OPT_DROP` puntos **menos de recall** que el umbral 0,5, en la fracción exigida de escenarios. Se muestra también el umbral que maximiza F1 como contraste.

In [ ]:
def veredicto_c3():
    hits, k = 0, len(scen)
    for name, tb in curves.items():
        i_acc = int(np.argmax(tb.accuracy.values))        # empate -> el umbral más bajo (el que más detecta)
        i_f1 = int(np.nanargmax(tb.f1.values))
        sel = pd.DataFrame({"criterio": ["umbral 0,5", "máximo accuracy", "máximo F1"],
                            **{c: [tb.loc[i05_, c] for i05_ in (I05, i_acc, i_f1)]
                               for c in ["umbral", "accuracy", "recall", "precision", "f1", "FP_por_millon", "FN_por_millon"]}})
        print(f"\n--- {name} ---")
        display(sel.round(4))
        drop = tb.loc[I05, "recall"] - tb.loc[i_acc, "recall"]
        ok = drop >= ACC_OPT_DROP
        hits += int(ok)
        print(f"Recall al optimizar accuracy: {tb.loc[i_acc, 'recall']:.3f} frente a {tb.loc[I05, 'recall']:.3f} con 0,5 (pérdida {drop:+.3f}) -> {'SE CUMPLE' if ok else 'no se cumple'}")
    ok_all = hits >= need_n(k)
    print(f"\nC3 se cumple en {hits}/{k} escenarios (se exigen {need_n(k)}) -> {'SE CUMPLE' if ok_all else 'NO SE CUMPLE'}")
    return ok_all

C3_OK = veredicto_c3()

print("\n================ VEREDICTO SOBRE LA HIPÓTESIS ================")
print(f"C1 (el accuracy no distingue al modelo del que no hace nada): {'sí' if C1_OK else 'no'}")
print(f"C2 (mismo accuracy ≥ 99 %, recall muy distinto):             {'sí' if C2_OK else 'no'}")
print(f"C3 (optimizar accuracy empuja al silencio) [secundaria]:      {'sí' if C3_OK else 'no'}")
if C1_OK and C2_OK:   print("-> HIPÓTESIS CONFIRMADA (C1 y C2)")
elif C1_OK or C2_OK:  print("-> HIPÓTESIS CONFIRMADA SOLO EN PARTE (una de C1/C2)")
else:                 print("-> HIPÓTESIS REFUTADA (ni C1 ni C2)")

## 5.1 Revisión exploratoria de C2 y C3 (añadida DESPUÉS de ver los datos)

Al ejecutar las secciones 4 y 5 con los datos reales, C1, C2 y C3 se cumplieron. Revisando **por qué**, aparecieron dos problemas de diseño en las reglas. Las reglas originales (C1, C2, C3) **se mantienen intactas y su veredicto sigue siendo el pre-registrado**. Lo que sigue son versiones revisadas, marcadas como **exploratorias** porque se fijaron viendo los datos: son una pista para entender, no una confirmación.

**Problema de C2.** Excluí el detector que *nunca* alerta (recall = 0), pero no los que alertan *casi nunca*. A prevalencia 1 %, cualquier umbral cercano a 1 tiene accuracy ≈ 99 % con recall de 0,01 o 0,02, así que «el recall varía mucho dentro del 99 %» es casi inevitable. Eso no prueba que el accuracy sea ciego a detectores *razonables*.
*Revisión (C2'):* se exige un suelo de recall (`C2B_RECALL_FLOOR`), y se muestra una **tabla de sensibilidad** con varios suelos, porque el suelo es una elección arbitraria y conviene ver cuánto depende de ella la conclusión.

**Dato útil para interpretar C2.** A prevalencia `p`, `accuracy ≥ 1 − p` equivale a `TP ≥ FP`, es decir, a **precisión ≥ 50 %**. Por tanto, *todo* punto con accuracy ≥ 99 % a `p = 1 %` ya tiene precisión ≥ 50 % por construcción. El rango de precisión de ese grupo no es un hallazgo, es una identidad (se comprueba abajo).

**Problema de C3.** Comparé el umbral que maximiza accuracy contra el umbral **0,5**. Pero 0,5 es un mal punto de operación a prevalencia 1 % (el modelo se entrenó con otra prevalencia), así que cualquier umbral mejor, sea del criterio que sea, pierde recall frente a él. La regla no aislaba el efecto del *accuracy*. Además, el umbral que maximiza accuracy coincidió (o casi) con el que maximiza F1.
*Revisión (C3'):* se compara el umbral que maximiza accuracy contra el que maximiza **F1**, que es el contraste justo: ambos son «umbrales buenos», y lo que se pregunta es si el accuracy empuja *más* hacia el silencio que una métrica que sí mira la detección. Se muestra además cómo evoluciona la diferencia al bajar la prevalencia.

### 5.1.1 C2' (exploratoria): mismo accuracy ≥ 99 %, pero solo entre detectores que detectan de verdad

In [ ]:
# --- comprobación de la identidad: accuracy >= 1-p  <=>  TP >= FP  <=>  precisión >= 0,5 ---
for name, tb in curves.items():
    G = tb[(tb.accuracy >= 1 - P_MAIN) & (tb.recall > 0)]
    assert (G.precision >= 0.5 - 1e-9).all(), "la identidad accuracy>=1-p  =>  precisión>=0,5 falla"
print("Identidad comprobada: todo umbral con accuracy >= 1 - p tiene precisión >= 50 %.\n")

# --- sensibilidad al suelo de recall ---
rows = []
for name, tb in curves.items():
    for fl in C2B_FLOORS_SENS:
        S = tb[(tb.accuracy >= ACC_MIRAGE) & (tb.recall >= fl) & (tb.recall > 0)]
        if len(S) < 2:
            rows.append(dict(escenario=name, suelo_recall=fl, n_umbrales=len(S)))
            continue
        lo, hi = S.loc[S.recall.idxmin()], S.loc[S.recall.idxmax()]
        rows.append(dict(escenario=name, suelo_recall=fl, n_umbrales=len(S),
                         recall_min=lo.recall, recall_max=hi.recall, rango=hi.recall - lo.recall,
                         FN_por_millon_en_recall_min=lo.FN_por_millon, FN_por_millon_en_recall_max=hi.FN_por_millon,
                         FP_por_millon_en_recall_min=lo.FP_por_millon, FP_por_millon_en_recall_max=hi.FP_por_millon))
tab_c2b = pd.DataFrame(rows)
print(f"Umbrales con accuracy >= {ACC_MIRAGE:.0%} a prevalencia {P_MAIN:.1%}, según el suelo de recall exigido:")
display(tab_c2b.round(4))


def veredicto_c2b():
    hits, k = 0, len(scen)
    for name, tb in curves.items():
        S = tb[(tb.accuracy >= ACC_MIRAGE) & (tb.recall >= C2B_RECALL_FLOOR)]
        if len(S) < 2:
            print(f"{name:>15}: solo {len(S)} umbral(es) con accuracy ≥ {ACC_MIRAGE:.0%} y recall ≥ {C2B_RECALL_FLOOR} -> no se puede evaluar (cuenta como no cumplida)")
            continue
        lo, hi = S.loc[S.recall.idxmin()], S.loc[S.recall.idxmax()]
        rng_ = hi.recall - lo.recall
        ok = rng_ >= RECALL_RANGE_BIG
        hits += int(ok)
        print(f"{name:>15}: {len(S)} umbrales | recall de {lo.recall:.3f} a {hi.recall:.3f} (rango {rng_:.3f}) | "
              f"ataques que se escapan por millón: {lo.FN_por_millon:,.0f} (umbral {lo.umbral:.4f}) frente a {hi.FN_por_millon:,.0f} (umbral {hi.umbral:.4f}) "
              f"-> {'SE CUMPLE' if ok else 'no se cumple'}")
    ok_all = hits >= need_n(k)
    print(f"\nC2' (exploratoria) se cumple en {hits}/{k} escenarios (se exigen {need_n(k)}) -> {'SE CUMPLE' if ok_all else 'NO SE CUMPLE'}")
    return ok_all

C2B_OK = veredicto_c2b()

### 5.1.2 C3' (exploratoria): optimizar accuracy frente a optimizar F1, según la prevalencia

Los dos son «umbrales buenos». Si el accuracy empujara de verdad hacia el silencio, su umbral sería **más alto** y su recall **más bajo** que los del F1, y la diferencia debería **crecer al bajar la prevalencia**.

In [ ]:
rows = []
for name, s in scen.items():
    for p in PREVALENCES:
        tb = curve_table(s, p)
        ia, i1 = int(np.argmax(tb.accuracy.values)), int(np.nanargmax(tb.f1.values))
        a, f = tb.iloc[ia], tb.iloc[i1]
        rows.append(dict(escenario=name, prevalencia=p,
                         umbral_acc=a.umbral, umbral_f1=f.umbral,
                         recall_acc=a.recall, recall_f1=f.recall, perdida_recall=f.recall - a.recall,
                         precision_acc=a.precision, precision_f1=f.precision,
                         FP_mill_acc=a.FP_por_millon, FP_mill_f1=f.FP_por_millon,
                         FN_mill_acc=a.FN_por_millon, FN_mill_f1=f.FN_por_millon))
tab_c3b = pd.DataFrame(rows)
display(tab_c3b.round(4))


def veredicto_c3b():
    hits, k = 0, len(scen)
    for name in scen:
        r = tab_c3b[(tab_c3b.escenario == name) & (np.isclose(tab_c3b.prevalencia, P_MAIN))].iloc[0]
        ok = r.perdida_recall >= ACC_OPT_DROP_VS_F1
        hits += int(ok)
        print(f"{name:>15}: umbral(acc) {r.umbral_acc:.3f} vs umbral(F1) {r.umbral_f1:.3f} | recall {r.recall_acc:.3f} vs {r.recall_f1:.3f} "
              f"(pérdida {r.perdida_recall:+.3f}) -> {'SE CUMPLE' if ok else 'no se cumple'}")
    ok_all = hits >= need_n(k)
    print(f"\nC3' (exploratoria) se cumple en {hits}/{k} escenarios (se exigen {need_n(k)}) -> {'SE CUMPLE' if ok_all else 'NO SE CUMPLE'}")
    return ok_all

C3B_OK = veredicto_c3b()

### 5.1.3 ¿Y con un umbral razonable? Cuánto mejora el modelo sobre «todo normal»

Los veredictos de C1 se calculan con el umbral 0,5. Aquí se mira el modelo en **su mejor umbral para accuracy** (elegido con las etiquetas de prueba, así que es una cota optimista). `errores_eliminados` = fracción de los errores de «todo normal» que el modelo elimina (1 = perfecto; ≤ 0 = no mejora).

Sirve para separar dos cosas que C1 mezcla: que el **accuracy** sea mala métrica, y que el **umbral 0,5** esté mal puesto para esa prevalencia.

In [ ]:
rows = []
for name, s in scen.items():
    for p in PREVALENCES:
        tb = curve_table(s, p)
        i = int(np.argmax(tb.accuracy.values))
        a = tb.iloc[i]
        rows.append(dict(escenario=name, prevalencia=p, umbral=a.umbral, acc_modelo=a.accuracy, acc_todo_normal=1 - p,
                         errores_eliminados=(a.accuracy - (1 - p)) / p, recall=a.recall, precision=a.precision,
                         FP_por_millon=a.FP_por_millon, FN_por_millon=a.FN_por_millon))
tab_good = pd.DataFrame(rows)
display(tab_good.round(4))

### 5.1.4 Resumen: pre-registro frente a revisión exploratoria

In [ ]:
resumen = pd.DataFrame([
    dict(regla="C1", estado="pre-registrada", se_cumple=C1_OK),
    dict(regla="C2", estado="pre-registrada", se_cumple=C2_OK),
    dict(regla="C3", estado="pre-registrada (secundaria)", se_cumple=C3_OK),
    dict(regla="C2'", estado="EXPLORATORIA (revisada tras ver datos)", se_cumple=C2B_OK),
    dict(regla="C3'", estado="EXPLORATORIA (revisada tras ver datos)", se_cumple=C3B_OK),
])
display(resumen)
print("El veredicto sobre la hipótesis es el PRE-REGISTRADO (C1 y C2). Las reglas con prima (') son exploratorias:")
print("sirven para entender cuánto depende la conclusión del diseño de las reglas, no para confirmar nada.")

## 6. Diagnósticos

### 6.1 ¿Falla el cálculo exacto? Verificación con re-muestreo real
Se construyen conjuntos de prueba reales con cada prevalencia y se mide el modelo (umbral 0,5) sobre ellos. Si el cálculo exacto es correcto, ambas cifras coinciden salvo ruido de muestreo.

In [ ]:
rows = []
for name, s in scen.items():
    yt, proba = s["first"]["y_te"], s["first"]["proba"]
    tpr_full, fpr_full = roc_points(yt, proba, np.array([THRESHOLD]))
    for p in PREVALENCES:
        vals = []
        for r in range(N_RESAMPLES):
            idx = resample_to_prevalence(yt, p, np.random.default_rng(SEED + 100 * r))
            vals.append(metrics_from_proba(yt[idx], proba[idx]))
        v = pd.DataFrame(vals)
        ex = at_prevalence(tpr_full[0], fpr_full[0], p)       # exacto con la MISMA primera repetición
        rows.append(dict(escenario=name, prevalencia=p,
                         acc_remuestreo=v.accuracy.mean(), acc_std=v.accuracy.std(), acc_exacto=ex["accuracy"],
                         prec_remuestreo=v.precision.mean(), prec_exacto=ex["precision"],
                         rec_remuestreo=v.recall.mean(), rec_exacto=ex["recall"],
                         max_diff=max(abs(v.accuracy.mean() - ex["accuracy"]), abs(v.precision.mean() - ex["precision"]),
                                      abs(v.recall.mean() - ex["recall"]))))
chk = pd.DataFrame(rows)
display(chk.round(4))
print(f"Diferencia máxima entre re-muestreo y cálculo exacto: {chk.max_diff.max():.4f}")

#### 6.1.1 Lo mismo, pero en otros umbrales (necesario para C2' y C3')
C2' y C3' usan umbrales distintos de 0,5. Se repite la verificación con re-muestreo real en varios umbrales y a la prevalencia principal, para comprobar que las curvas que usan esas reglas también son fiables.

In [ ]:
rows = []
for name, s in scen.items():
    yt, proba = s["first"]["y_te"], s["first"]["proba"]
    for thr in [0.1, 0.5, 0.9, 0.99, 0.999]:
        tpr_f, fpr_f = roc_points(yt, proba, np.array([thr]))
        ex = at_prevalence(tpr_f[0], fpr_f[0], P_MAIN)
        vals = []
        for r in range(N_RESAMPLES):
            idx = resample_to_prevalence(yt, P_MAIN, np.random.default_rng(SEED + 100 * r))
            vals.append(metrics_from_proba(yt[idx], proba[idx], thr=thr))
        v = pd.DataFrame(vals)
        rows.append(dict(escenario=name, umbral=thr,
                         acc_remuestreo=v.accuracy.mean(), acc_exacto=ex["accuracy"],
                         rec_remuestreo=v.recall.mean(), rec_exacto=ex["recall"],
                         prec_remuestreo=v.precision.mean(), prec_exacto=ex["precision"],
                         max_diff=np.nanmax([abs(v.accuracy.mean() - ex["accuracy"]), abs(v.recall.mean() - ex["recall"]),
                                             abs(v.precision.mean() - ex["precision"])])))
chk_thr = pd.DataFrame(rows)
display(chk_thr.round(4))
print(f"Diferencia máxima entre re-muestreo y cálculo exacto en otros umbrales: {chk_thr.max_diff.max():.4f}")
print("Nota: las diferencias de recall y precisión son ruido de muestreo y crecen cuando hay pocos ataques en el re-muestreo (o pocas alertas, en umbrales altos).")
print("El accuracy, que es lo que más usan las reglas, es el que menos se desvía.")

### 6.2 Cuándo gana el modelo al detector que no hace nada
Como `Δ = TP − FP`, el modelo supera a «todo normal» en accuracy **si y solo si** detecta más ataques que falsas alarmas lanza, es decir, si su **precisión supera el 50 %**. Eso ocurre a partir de una **prevalencia crítica** `p* = FPR / (TPR + FPR)`. Por debajo de `p*`, el detector que no hace nada saca mejor accuracy.

También se muestra la matriz de confusión **por millón de conexiones** a prevalencia `P_MAIN`, que es lo que el equipo de un SOC vería en operación.

In [ ]:
rows = []
for name, s in scen.items():
    tpr, fpr = s["tpr"][I05], s["fpr"][I05]
    p_star = fpr / (tpr + fpr) if (tpr + fpr) > 0 else np.nan     # sin alertas de ningún tipo: no existe
    # comprobación numérica: en p* el accuracy del modelo coincide con el de "todo normal" y la precisión es 0,5.
    # Solo tiene sentido si el modelo alerta alguna vez y se equivoca alguna vez (0 < p* < 1).
    if 0 < p_star < 1:
        m = at_prevalence(tpr, fpr, p_star)
        assert abs(m["accuracy"] - (1 - p_star)) < 1e-12 and abs(m["precision"] - 0.5) < 1e-12
    else:
        print(f"Aviso ({name}): TPR={tpr:.4f}, FPR={fpr:.4f} -> p* degenerada ({p_star}); el modelo casi no alerta con el umbral {THRESHOLD}.")
    rows.append(dict(escenario=name, TPR=tpr, FPR=fpr, prevalencia_critica=p_star,
                     prevalencia_dataset_prueba=s["n_test_pos"] / s["n_test"]))
display(pd.DataFrame(rows).round(5))

print(f"\nMatriz de confusión por 1.000.000 de conexiones (prevalencia {P_MAIN:.1%}, umbral {THRESHOLD}):")
for name, s in scen.items():
    m = at_prevalence(s["tpr"][I05], s["fpr"][I05], P_MAIN)
    mat = pd.DataFrame({"alerta": [m["tp_mill"], m["fp_mill"]], "sin alerta": [m["fn_mill"], m["tn_mill"]]},
                       index=["ataque real", "tráfico normal"]).round(0).astype(int)
    print(f"\n--- {name} ---")
    display(mat)
    print(f"Alertas totales: {m['tp_mill'] + m['fp_mill']:,.0f} | de ellas son ataques: {m['precision']:.1%} | "
          f"ataques que se escapan: {m['fn_mill']:,.0f} de {P_MAIN * 1e6:,.0f}")

## 7. Figuras

In [ ]:
FIG_DIR.mkdir(exist_ok=True)
ps = np.logspace(-3, np.log10(0.5), 200)
colors = {"tiempo (A → B)": "tab:red", "azar": "tab:blue"}

# Figura 1: accuracy del modelo y de "todo normal" según la prevalencia
fig, ax = plt.subplots(figsize=(7.5, 4.6))
ax.plot(ps, 1 - ps, "k--", label="«todo normal» (no detecta nada)")
for name, s in scen.items():
    ax.plot(ps, [at_prevalence(s["tpr"][I05], s["fpr"][I05], p)["accuracy"] for p in ps], color=colors[name],
            label=f"modelo, umbral 0,5 · {name}")
ax.axhline(ACC_MIRAGE, color="gray", lw=0.8, ls=":")
ax.set_xscale("log"); ax.set_xlabel("prevalencia de ataques en la prueba"); ax.set_ylabel("accuracy")
ax.set_title("El accuracy depende de cuántos ataques haya, no solo del modelo"); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(FIG_DIR / "fig1_accuracy_vs_prevalencia.png", dpi=150); plt.show()

# Figura 2: precisión y recall según la prevalencia
fig, ax = plt.subplots(figsize=(7.5, 4.6))
for name, s in scen.items():
    prec = [at_prevalence(s["tpr"][I05], s["fpr"][I05], p)["precision"] for p in ps]
    ax.plot(ps, prec, color=colors[name], label=f"precisión · {name}")
    ax.axhline(s["tpr"][I05], color=colors[name], ls=":", lw=1, label=f"recall (no cambia) · {name}")
ax.axhline(0.5, color="gray", lw=0.8)
ax.set_xscale("log"); ax.set_xlabel("prevalencia de ataques en la prueba"); ax.set_ylabel("valor de la métrica")
ax.set_title("Con pocos ataques, la precisión se hunde y el recall no se entera"); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(FIG_DIR / "fig2_precision_recall_vs_prevalencia.png", dpi=150); plt.show()

# Figura 3: accuracy frente a recall al variar el umbral (prevalencia principal)
fig, ax = plt.subplots(figsize=(7.5, 4.6))
for name, tb in curves.items():
    ax.plot(tb.recall, tb.accuracy, color=colors[name], label=name)
    ax.scatter([tb.loc[I05, "recall"]], [tb.loc[I05, "accuracy"]], color=colors[name], marker="*", s=120, zorder=3)
ax.axhline(ACC_MIRAGE, color="gray", lw=0.8, ls=":")
ax.set_xlabel("recall"); ax.set_ylabel("accuracy")
ax.set_title(f"Misma línea de accuracy, recall muy distinto (prevalencia {P_MAIN:.0%}; ★ = umbral 0,5)"); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(FIG_DIR / "fig3_accuracy_vs_recall_umbrales.png", dpi=150); plt.show()

### 7.1 Exportar las tablas para estudiarlas

Guarda en `resultados/` las tablas principales en CSV (abribles con Excel o pandas), para poder revisar los números sin volver a ejecutar el cuaderno.

In [ ]:
RES_DIR.mkdir(exist_ok=True)
outputs = {
    "01_metricas_por_prevalencia.csv": tab_prev,
    "02_diagnostico_remuestreo.csv": chk,
    "02b_diagnostico_remuestreo_otros_umbrales.csv": chk_thr,
    "03_c2_sensibilidad_suelo_recall.csv": tab_c2b,
    "04_c3_acc_vs_f1_por_prevalencia.csv": tab_c3b,
    "05_umbral_razonable_vs_todo_normal.csv": tab_good,
    "06_resumen_reglas.csv": resumen,
}
for name, tb in curves.items():
    outputs[f"07_curva_umbrales_p1pct_{'tiempo' if 'tiempo' in name else 'azar'}.csv"] = tb
for fname, tb in outputs.items():
    tb.to_csv(RES_DIR / fname, index=False)
print("Guardado en", RES_DIR.resolve())
for fname in outputs:
    print(" -", fname)

## 8. Conclusiones (guía para rellenar)

Rellena esto **después** de ejecutar, en el informe (`README.md`), separando lo planeado de lo descubierto:

1. **Veredicto de C1, C2 y C3** (pre-registrados), con los números de las secciones 3.2, 4 y 5. Después, **por separado y como exploratorio**, C2' y C3' y la tabla 5.1.3. ¿Coinciden con las predicciones del informe (sección 5.6)?
2. **El mismo modelo, dos notas.** Compara `acc_modelo` y `acc_todo_normal` a prevalencia 1 % en la tabla de 3.1.
3. **FN y FP en cifras absolutas** por millón de conexiones (sección 6.2). Un 99 % de accuracy con 1 % de ataques sigue siendo decenas de miles de errores.
4. **Prevalencia crítica** `p*` de cada escenario.
5. Todo lo que añadas tras ver los datos, márcalo como **exploratorio**.
6. Anota en la tabla de errores lo que se haya roto durante la ejecución.